# Step 1: SQL Models (Linked to Notebook Sections)

📘 SQL Model 1 — `core__guest_purchase_summary.sql`

**Purpose**: Core behavioral profile for each resolved guest: total spend, visit count, recency.

In [5]:
sql1="""
-- MODEL: core__guest_purchase_summary.sql
-- PURPOSE: Summarizes each guest's transactional behavior using resolved identity stitching
-- Feeds segment tagging, visit gap analysis, and targeting logic in NBO EDA.

WITH base_txns AS (
    SELECT
        -- ⚙ Use final transactional table from Subway Redshift environment
        t.resolved_guest_id, -- pre-stitched ID via payment_token or hashed identity logic
        t.transaction_id,
        t.amount,
        t.purchase_date::DATE,
        t.channel,
        t.offer_code,
        t.store_id
    FROM raw.transactions t
    WHERE t.transaction_status = 'completed'
      AND t.amount IS NOT NULL
),
guest_summary AS (
    SELECT
        resolved_guest_id,
        COUNT(DISTINCT transaction_id) AS total_transactions,
        COUNT(DISTINCT purchase_date) AS visit_count,
        SUM(amount) AS total_spend,
        MIN(purchase_date) AS first_seen,
        MAX(purchase_date) AS last_seen,
        DATEDIFF('day', MIN(purchase_date), MAX(purchase_date)) AS days_between_visits
    FROM base_txns
    GROUP BY resolved_guest_id
)
SELECT * FROM guest_summary;
"""

📘 SQL Model 2 — `core__guest_segments.sql`

**Purpose**: Apply business logic for segment classification (one-and-done, occasional, etc.)

In [ ]:
sql2="""
-- MODEL: core__guest_segments.sql
-- PURPOSE: Assign guest segments based on behavioral cutoffs aligned with strategy logic

WITH summary AS (
    SELECT * FROM {{ ref('core__guest_purchase_summary') }}
)
SELECT
    resolved_guest_id,
    total_transactions,
    total_spend,
    visit_count,
    first_seen,
    last_seen,
    days_between_visits,
    -- 🎯 Segment classification logic
    CASE
        WHEN visit_count = 1 THEN 'one_and_done'
        WHEN total_spend >= 95 THEN 'high_value'
        WHEN total_spend >= 40 THEN 'occasional'
        ELSE 'low_value'
    END AS segment
FROM summary;
"""

📘 SQL Model 3 — `derived__return_lag_model.sql`

**Purpose**: Creates inter-visit lag table per user for dropoff prediction

In [ ]:
sql3="""
-- MODEL: derived__return_lag_model.sql
-- PURPOSE: Calculates days between visits to feed return lag histograms and drop-off modeling

WITH base_txns AS (
    SELECT
        resolved_guest_id,
        purchase_date::DATE
    FROM raw.transactions
    WHERE transaction_status = 'completed'
),
ranked_txns AS (
    SELECT
        resolved_guest_id,
        purchase_date,
        ROW_NUMBER() OVER (PARTITION BY resolved_guest_id ORDER BY purchase_date) AS rn
    FROM base_txns
),
return_lags AS (
    SELECT
        t1.resolved_guest_id,
        DATEDIFF('day', t1.purchase_date, t2.purchase_date) AS return_lag_days
    FROM ranked_txns t1
    JOIN ranked_txns t2
      ON t1.resolved_guest_id = t2.resolved_guest_id
     AND t1.rn = t2.rn - 1
)
SELECT * FROM return_lags;
"""

📘 SQL Model 4 — `derived__channel_behavior_summary.sql`

**Purpose**: Measures channel usage distribution by segment for funnel attribution

In [ ]:
sql4="""
-- MODEL: derived__channel_behavior_summary.sql
-- PURPOSE: Calculates channel-wise behavior normalized by segment

WITH joined AS (
    SELECT
        s.segment,
        t.resolved_guest_id,
        t.channel
    FROM {{ ref('core__guest_segments') }} s
    JOIN raw.transactions t
      ON s.resolved_guest_id = t.resolved_guest_id
    WHERE t.transaction_status = 'completed'
)
SELECT
    segment,
    channel,
    COUNT(DISTINCT resolved_guest_id) AS guest_count,
    ROUND(
        COUNT(DISTINCT resolved_guest_id) * 1.0
        / SUM(COUNT(DISTINCT resolved_guest_id)) OVER (PARTITION BY segment),
        4
    ) AS normalized_share
FROM joined
GROUP BY segment, channel;
"""

📘 SQL Model 5 — `derived__recency_offer_bins.sql`

**Purpose**: Bin users by time-since-last-visit to determine offer intervention windows

In [4]:
sql5="""
-- MODEL: derived__recency_offer_bins.sql
-- PURPOSE: Bins guests by days since last visit for offer strength logic

WITH recent AS (
    SELECT
        resolved_guest_id,
        MAX(purchase_date) AS last_seen
    FROM raw.transactions
    WHERE transaction_status = 'completed'
    GROUP BY resolved_guest_id
)
SELECT
    resolved_guest_id,
    DATEDIFF('day', last_seen, CURRENT_DATE) AS days_since_last_purchase,
    CASE
        WHEN DATEDIFF('day', last_seen, CURRENT_DATE) < 7 THEN '[0–7)'
        WHEN DATEDIFF('day', last_seen, CURRENT_DATE) < 14 THEN '[7–14)'
        WHEN DATEDIFF('day', last_seen, CURRENT_DATE) < 21 THEN '[14–21)'
        WHEN DATEDIFF('day', last_seen, CURRENT_DATE) < 30 THEN '[21–30)'
        WHEN DATEDIFF('day', last_seen, CURRENT_DATE) < 60 THEN '[30–60)'
        WHEN DATEDIFF('day', last_seen, CURRENT_DATE) < 90 THEN '[60–90)'
        ELSE '[90–180)'
    END AS offer_timing_bin
FROM recent;
"""